# Лабораторная работа №1

#Датасет: Pokemon.csv

## 1. Постановка задачи

Данные — это таблица с покемонами из игр Pokemon (поколения 1-6). В каждой строке один покемон или его форма (например, мега-эволюция). Есть имя, типы, шесть базовых характеристик (HP, Attack, Defense, Sp. Atk, Sp. Def, Speed), суммарный Total, номер поколения и флаг Legendary.

Заказчик — условно игровая студия или аналитики по киберспорту. Им интересно: какие группы покемонов есть, чем легендарные отличаются от обычных, какие покемоны выбиваются из общего ряда.

Задачи, которые можно решать:
- кластеризация покемонов по характеристикам;
- предсказание Legendary по статам;
- поиск выбросов;
- сравнение поколений и типов между собой.

Поколение (Generation) — это по сути временной признак, они выходили по порядку: 1996, 1999, 2002, 2006, 2010, 2013.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

df = pd.read_csv(Path("../data/pokemon.csv"))
df.columns = df.columns.str.strip()
df.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


## 2. Паспорт датасета

In [3]:
print("Строк:", df.shape[0])
print("Столбцов:", df.shape[1])

Строк: 800
Столбцов: 13


In [4]:
df.dtypes

#             int64
Name            str
Type 1          str
Type 2          str
Total         int64
HP            int64
Attack        int64
Defense       int64
Sp. Atk       int64
Sp. Def       int64
Speed         int64
Generation    int64
Legendary      bool
dtype: object

In [5]:
df["Legendary"] = df["Legendary"].astype("category")
df["Type 1"] = df["Type 1"].astype("category")
df["Type 2"] = df["Type 2"].astype("category")
df["Generation"] = df["Generation"].astype("category")

years = {1: 1996, 2: 1999, 3: 2002, 4: 2006, 5: 2010, 6: 2013}
df["release_year"] = pd.to_datetime(df["Generation"].astype(int).map(years), format="%Y")

df.dtypes

#                        int64
Name                       str
Type 1                category
Type 2                category
Total                    int64
HP                       int64
Attack                   int64
Defense                  int64
Sp. Atk                  int64
Sp. Def                  int64
Speed                    int64
Generation            category
Legendary             category
release_year    datetime64[us]
dtype: object

In [6]:
info = pd.DataFrame({
    "тип": df.dtypes.astype(str),
    "пропуски": df.isna().sum(),
    "уникальных": df.nunique(),
})
info

,тип,пропуски,уникальных
#,int64,0,721
Name,str,0,800
Type 1,category,0,18
Type 2,category,386,18
Total,int64,0,200
HP,int64,0,94
Attack,int64,0,111
Defense,int64,0,103
Sp. Atk,int64,0,105
Sp. Def,int64,0,92


Признаки по смыслу:
- # — номер покемона в покедексе
- Name — имя (иногда с формой)
- Type 1 / Type 2 — боевые типы
- Total — сумма шести статов
- HP, Attack, Defense, Sp. Atk, Sp. Def, Speed — базовые характеристики
- Generation — поколение (1-6)
- Legendary — легендарный или нет

## 3. Аудит качества данных

### 3.1 Пропуски

In [7]:
miss = pd.DataFrame({
    "пропусков": df.isna().sum(),
    "процент": (df.isna().mean() * 100).round(2),
}).sort_values("процент", ascending=False)

miss

,пропусков,процент
Type 2,386,48.25
#,0,0.00
Name,0,0.00
Type 1,0,0.00
Total,0,0.00
HP,0,0.00
Attack,0,0.00
Defense,0,0.00
Sp. Atk,0,0.00
Sp. Def,0,0.00


Пропуски только в Type 2 — примерно у половины покемонов второго типа просто нет. Это нормально, не ошибка. Удалять такие строки нельзя, иначе потеряем половину данных. Логично заменить на "None" как отдельную категорию.

### 3.2 Дубликаты

In [8]:
print("Полных дубликатов:", df.duplicated().sum())
print("Дубликатов по #:", df["#"].duplicated().sum())
print("Дубликатов по Name:", df["Name"].duplicated().sum())

Полных дубликатов: 0
Дубликатов по #: 79
Дубликатов по Name: 0


In [9]:
dup = df[df["#"].duplicated(keep=False)]
dup[["#", "Name", "Type 1", "Generation"]].head(10)

,#,Name,Type 1,Generation
2,3,Venusaur,Grass,1
3,3,VenusaurMega Venusaur,Grass,1
6,6,Charizard,Fire,1
7,6,CharizardMega Charizard X,Fire,1
8,6,CharizardMega Charizard Y,Fire,1
11,9,Blastoise,Water,1
12,9,BlastoiseMega Blastoise,Water,1
18,15,Beedrill,Bug,1
19,15,BeedrillMega Beedrill,Bug,1
22,18,Pidgeot,Normal,1


Полных дубликатов строк нет. Но номера (#) повторяются — это мега-эволюции и альтернативные формы (например, №6 — это Charizard и два его мега-вида). Это не ошибка, просто # не является уникальным ключом.

### 3.3 Числовые признаки

In [ ]:
num = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed", "Total"]
df[num].describe().round(2)

In [ ]:
for c in num:
    print(c, "| min:", df[c].min(), "| max:", df[c].max(), "| отрицательных:", (df[c] < 0).sum())

Отрицательных значений нет, всё в диапазоне [1; 255]. Единственное, что бросается в глаза — Shedinja с HP = 1, но это фича персонажа, а не ошибка данных. Total — это сумма остальных статов, так что с ним надо аккуратно (мультиколлинеарность).

### 3.4 Категориальные признаки

In [ ]:
for c in ["Type 1", "Type 2", "Generation", "Legendary"]:
    print("---", c, "---")
    print(df[c].value_counts(dropna=False).head(8))
    print()

In [ ]:
for c in ["Type 1", "Type 2", "Generation"]:
    s = df[c].astype(str)
    clean = s.str.strip().str.lower()
    if clean.nunique() != s.nunique():
        print(c, "— есть различия после нормализации")
    else:
        print(c, "— ок")

In [ ]:
merged = df[df["Name"].str.contains(r"[a-z][A-Z]", regex=True, na=False)]
print("Таких имён:", len(merged))
merged[["#", "Name"]].head(10)

В Type 1, Type 2, Generation — грязных категорий нет, всё чисто. А вот в Name есть слипшиеся имена форм типа "VenusaurMega Venusaur", "CharizardMega Charizard X". Это не критично для анализа, но если понадобится — можно вытащить форму регуляркой в отдельный столбец.

### 3.5 Выбросы

In [ ]:
def iqr_out(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return series[(series < low) | (series > high)]

for c in ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]:
    o = iqr_out(df[c])
    print(f"{c:8s} | выбросов: {len(o):3d} ({len(o)/len(df)*100:.1f}%)")

In [ ]:
col = "Defense"
out = iqr_out(df[col])
df.loc[out.index, ["#", "Name", col]].sort_values(col, ascending=False).head(10)

In [ ]:
sns.boxplot(x=df[col])
plt.title(f"Boxplot: {col}")
plt.show()

Выбросы есть во всех статах, но это не ошибки — так задумано. Blissey с HP 255, Shuckle с Defense 230, Deoxys-Speed со Speed 180 — это реальные игровые значения. Удалять их не стоит, но при обучении линейных моделей лучше использовать что-то устойчивое к выбросам или добавлять флаг is_outlier.

## 4. Разведочный анализ

In [ ]:
sns.histplot(df["Attack"], kde=True)
plt.title("Распределение Attack")
plt.xlabel("Attack")
plt.show()

Распределение Attack близко к нормальному, центр около 75-80. Справа длинный хвост — это легендарные и мега-покемоны. Похоже, что по одной атаке легендарных не отделить, но в комбинации с другими статами должно получиться.

In [ ]:
df["Type 1"].value_counts().plot(kind="bar")
plt.title("Покемоны по основному типу")
plt.ylabel("Количество")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Самые частые типы — Water, Normal, Grass. Редкий как основной — Flying (он чаще идёт вторичным). Возможно, у разных типов отличаются средние статы — можно проверить.

In [ ]:
sns.scatterplot(data=df, x="Attack", y="Defense", hue="Legendary", alpha=0.7)
plt.title("Attack vs Defense")
plt.show()

Легендарные (красные) собираются в правом верхнем углу: и атака, и защита высокие. Обычные так высоко почти не заходят. Похоже, эти два признака неплохо разделяют классы.

In [ ]:
sns.boxplot(data=df, x="Generation", y="Total")
plt.title("Total по поколениям")
plt.show()

Медиана Total по поколениям примерно одинаковая, но в 5-6 поколениях появляются более длинные хвосты — там больше мега-эволюций и легендарных. Возможно, со временем добавляют более сильных покемонов.

## Выводы

- Данные в целом чистые, серьёзных ошибок нет.
- Пропуски только в Type 2 — структурные, заменяем на "None".
- Дубликаты по # — это формы и мега-эволюции, не удаляем.
- Выбросы по статам — часть игровой механики.
- Легендарные хорошо отделяются по Attack и Defense.
- Total лучше не использовать вместе с остальными статами.